# AELIONIX BLACKFORGE — Phase 3 Colab Validation

This notebook performs a deterministic, one-click validation of the Blackforge **Evidence ↔ Memory Integration & Evidence Lifecycle** (Phase 3).

**What this validates:**
- Repository integrity and commit verification
- Dependency installation (runtime + dev extras)
- All Blackforge imports, including the new evidence modules
- Full automated test suite (evidence lifecycle + bridge included)
- Bootstrap + health verification (`evidence_store_ready`, `evidence_memory_link_ready`)
- **No fake authority**: LLM claims enter as `HYPOTHESIZED`; `VALIDATED` is only reachable through an authorized validation workflow
- Epistemic status transitions (legal moves only; downgrades rejected)
- Lifecycle operations (`SUPERSEDED` / `INVALIDATED` / `ARCHIVED`) that never rewrite epistemic history
- Typed evidence relationships (supports/contradicts/validates/supersedes/corroborates/…)
- Contradiction handling — nothing is deleted; both records persist with a typed link
- Deterministic deduplication of identical evidence
- Audited, status-independent confidence changes
- Evidence ↔ memory materialization and reverse lookups (persistent)
- Restart persistence for evidence, relationships AND linked memory records
- Transaction/compensation boundary documented and exercised

**What this does NOT do:**
- No model download or LLM inference (Phase 3 is stdlib-only — SQLite + pydantic)
- No offensive security actions, reconnaissance, or scanning
- No autonomous attack planning

**Runtime:** Google Colab (CPU or GPU) — the notebook runs identically on free CPU runtimes.

---
## 1. Runtime Information

In [ ]:
import sys
import platform

print("Blackforge Phase 3 Colab Validation (Evidence <-> Memory Integration)")
print("=" * 60)
print("Python:", sys.version.split()[0])
print("Executable:", sys.executable)
print("Platform:", platform.platform())
print("Architecture:", platform.machine())
print("=" * 60)

assert sys.version_info >= (3, 10), f"Blackforge requires Python 3.10+, got {sys.version}"
print("Python version check: PASS")

---
## 2. Repository Acquisition

In [ ]:
from pathlib import Path
import subprocess

# ── Configuration (edit here if fork changes) ──────────────────────────
REPO_URL = "https://github.com/Sagelord00000001/Blackforge.git"
REPO_DIR = Path("/content/blackforge")
# ───────────────────────────────────────────────────────────────────────

if REPO_DIR.exists() and (REPO_DIR / "blackforge" / "__init__.py").exists():
    print(f"Repository already exists at {REPO_DIR}, updating...")
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=False)
else:
    if REPO_DIR.exists():
        import shutil
        shutil.rmtree(REPO_DIR)
    subprocess.run(
        ["git", "clone", REPO_URL, str(REPO_DIR)],
        check=True,
    )

import os
os.chdir(str(REPO_DIR))
print(f"Repository ready at {REPO_DIR}")

---
## 3. Commit Verification

In [ ]:
import subprocess

EXPECTED_PHASE1_COMMIT = "1e54de4"

result = subprocess.run(
    ["git", "-C", str(REPO_DIR), "log", "-1", "--format=%h"],
    capture_output=True, text=True, check=True,
)
current_commit = result.stdout.strip()

print(f"Expected Phase 1 baseline: {EXPECTED_PHASE1_COMMIT}")
print(f"Current repository commit: {current_commit}")

result_log = subprocess.run(
    ["git", "-C", str(REPO_DIR), "log", "--oneline"],
    capture_output=True, text=True, check=True,
)
commits = [line.split()[0] for line in result_log.stdout.strip().splitlines()]

has_phase3_evidence = (
    (REPO_DIR / "blackforge" / "evidence" / "repository.py").exists()
    and (REPO_DIR / "blackforge" / "evidence" / "bridge.py").exists()
)
has_phase2_memory = (REPO_DIR / "blackforge" / "memory" / "repository.py").exists()

if not has_phase3_evidence:
    raise RuntimeError("Phase 3 evidence modules not present — repo is ahead or behind.")
print("Phase 3 evidence modules present: PASS")

if EXPECTED_PHASE1_COMMIT in commits:
    print("Commit verification: PASS (Phase 1 baseline found)")
elif any(c.startswith(EXPECTED_PHASE1_COMMIT[:4]) for c in commits):
    print("Commit verification: PASS (Phase 1 baseline found, abbreviated match)")
else:
    result_merge = subprocess.run(
        ["git", "-C", str(REPO_DIR), "merge-base", "--is-ancestor",
         EXPECTED_PHASE1_COMMIT, current_commit],
        capture_output=True, check=False,
    )
    if result_merge.returncode == 0:
        print("Commit verification: PASS (repo advanced past Phase 1)")
    else:
        print(f"WARNING: Phase 1 commit {EXPECTED_PHASE1_COMMIT} not found in history.")
        print("The repo may predate Phase 1. Continuing anyway.")

---
## 4. Install Blackforge

Phase 3 (evidence + memory) uses only the Python standard library plus the existing runtime. Only the `[dev]` extra is installed — the `llm` extra (torch/transformers) is intentionally left out, so this notebook is lightweight and OOM-free on free CPU runtimes.

In [ ]:
!pip install hatchling --quiet
!pip install -e ".[dev]"

import blackforge
print("Blackforge import: PASS")

---
## 5. Environment / Import Health Check

In [ ]:
import importlib

modules = [
    "blackforge",
    "blackforge.core.config",
    "blackforge.core.errors",
    "blackforge.core.types",
    "blackforge.core.logging",
    "blackforge.runtime.bootstrap",
    "blackforge.runtime.hardware",
    "blackforge.memory",
    "blackforge.memory.base",
    "blackforge.memory.provenance",
    "blackforge.memory.repository",
    "blackforge.memory.manager",
    "blackforge.evidence",
    "blackforge.evidence.models",
    "blackforge.evidence.rules",
    "blackforge.evidence.query",
    "blackforge.evidence.repository",
    "blackforge.evidence.store",
    "blackforge.evidence.bridge",
    "blackforge.mission.manager",
    "blackforge.capabilities.registry",
    "blackforge.authorization",
    "blackforge.scope.validator",
]

_import_failures = []
for module in modules:
    try:
        importlib.import_module(module)
    except Exception as e:
        _import_failures.append((module, str(e)))

if _import_failures:
    for mod, err in _import_failures:
        print(f"  FAIL: {mod} — {err}")
    raise RuntimeError(f"Import health check failed: {len(_import_failures)} module(s)")

print(f"Blackforge imports OK ({len(modules)} modules verified).")
print("Evidence + memory module imports: PASS")

---
## 6. Automated Regression Tests

Runs the full suite including `tests/test_evidence_phase3.py`. The LLM/torch-heavy files are excluded: importing the HF provider pulls ~2GB of torch memory and can SIGKILL the kernel on CPU runtimes.

In [ ]:
import subprocess
import sys

print("Running automated test suite...")
result = subprocess.run(
    [
        sys.executable, "-m", "pytest", "-q", "--tb=short",
        "--ignore=tests/test_huggingface_provider.py",
        "--ignore=tests/test_loader.py",
        "--ignore=tests/test_smoke_real_model.py",
    ],
    capture_output=True, text=True, cwd=str(REPO_DIR),
)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr[-500:] if result.stderr else "")
    raise RuntimeError(f"pytest failed with exit code {result.returncode}")

print("Automated test suite: PASS")

---
## 7. Bootstrap Verification

Bootstraps the app and confirms every subsystem — including `evidence_store_ready` and `evidence_memory_link_ready` — reports healthy.

In [ ]:
from blackforge.runtime.bootstrap import bootstrap

app = bootstrap()
assert app.healthy(), "Blackforge health check failed"
verification = app.verify()
assert verification["evidence_store_ready"], "evidence_store_ready must be True"
assert verification["evidence_memory_link_ready"], "evidence_memory_link_ready must be True"
assert verification["memory_ready"], "memory_ready must be True"

for k, v in verification.items():
    symbol = "PASS" if v else "FAIL"
    print(f"  [{symbol}] {k}")

print("\nBlackforge bootstrap (evidence + memory linked): PASS")

---
## 8. Evidence Rules — No Fake Authority & Status Transitions

LLM/analysis output must never become `VALIDATED` on its own. Claims enter as `HYPOTHESIZED`; only an authorized validation workflow creates `VALIDATED` records, which are linked back to the claim they confirm.

In [ ]:
from blackforge.core.types import EvidenceID, EvidenceStatus, EvidenceType, Confidence, MissionID, SessionID
from blackforge.core.errors import EvidenceRuleError
from blackforge.evidence.models import Evidence, EvidenceLifecycle, EvidenceRelation
from blackforge.evidence.store import EvidenceStore
from blackforge.evidence.query import EvidenceQuery

MID = MissionID("mission_phase3")
SID = SessionID("sess_phase3")

rules_store = EvidenceStore()

# 1) An LLM-style claim starts as a hypothesis, never as authority.
claim = rules_store.add_claim(MID, "endpoint /users may expose user profile data", session_id=SID)
assert claim.status == EvidenceStatus.HYPOTHESIZED, f"Expected hypothesized, got {claim.status}"
print(f"LLM claim enters as HYPOTHESIZED (id={claim.id}) — no fake authority: PASS")

# 2) Creating VALIDATED evidence outside a validation workflow is rejected.
rejected = False
try:
    rules_store.add(
        Evidence(
            mission_id=MID, source_capability="llm_inference", target="example.com",
            evidence_type=EvidenceType.OBSERVATION, status=EvidenceStatus.VALIDATED,
            raw_data="pretend this is authoritative",
        )
    )
except EvidenceRuleError:
    rejected = True
assert rejected, "VALIDATED creation without a validation workflow must be rejected"
print("Direct VALIDATED creation rejected: PASS")

# 3) The validation workflow is the only path to VALIDATED, and it links back.
validation = rules_store.add_validation(
    MID, "example.com /users",
    "authorized application-behavior analysis confirmed profile data in responses",
    source_capability="authorized_analysis", validates_id=claim.id, session_id=SID,
)
assert validation.status == EvidenceStatus.VALIDATED
validates = rules_store.related_evidence(claim.id, EvidenceRelation.VALIDATES)
assert any(x.evidence.id == validation.id for x in validates)
print("Validation workflow -> VALIDATED + VALIDATES link: PASS")

# 4) Legal transitions; downgrades rejected (lifecycle handles disproof).
obs = rules_store.add(
    Evidence(
        mission_id=MID, source_capability="port_scanner", target="example.com",
        evidence_type=EvidenceType.OBSERVATION, status=EvidenceStatus.OBSERVED,
        raw_data="443 open",
    )
)
rules_store.transition_status(obs.id, EvidenceStatus.INFERRED)
assert rules_store.get(obs.id).status == EvidenceStatus.INFERRED
downgrade_rejected = False
try:
    rules_store.transition_status(obs.id, EvidenceStatus.OBSERVED)
except EvidenceRuleError:
    downgrade_rejected = True
assert downgrade_rejected
print("Legal transition OBSERVED->INFERRED ok; illegal downgrade rejected: PASS")

# 5) Confidence is independent of status and fully audited.
rules_store.adjust_confidence(obs.id, Confidence.HIGH, reason="corroborated by a second sweep")
loaded_obs = rules_store.get(obs.id)
assert loaded_obs.confidence == Confidence.HIGH
assert loaded_obs.status == EvidenceStatus.INFERRED
assert len(loaded_obs.confidence_changes) == 1
print("Confidence raised WITHOUT touching status; change recorded in history: PASS")

print("Evidence rules engine: PASS")

---
## 9. Evidence Lifecycle, Relationships, Contradiction & Dedup

Lifecycle operations never rewrite the epistemic record. Contradiction never deletes — it links. Identical evidence is de-duplicated deterministically.

In [ ]:
def ev(raw, cap="scanner", etype=EvidenceType.OBSERVATION):
    return Evidence(
        mission_id=MID, session_id=SID, source_capability=cap, target="example.com",
        evidence_type=etype, status=EvidenceStatus.OBSERVED, raw_data=raw,
    )

lc_store = EvidenceStore()

# Contradiction: both records survive, typed CONTRADICTS link is recorded.
open_record = lc_store.add(ev("port 443 open"))
closed_record = ev("port 443 closed")
lc_store.contradict(open_record.id, closed_record, supersede=False)
assert lc_store.get(open_record.id).lifecycle == EvidenceLifecycle.ACTIVE
assert lc_store.get(closed_record.id).lifecycle == EvidenceLifecycle.ACTIVE
contra = lc_store.get_relationships(open_record.id)
assert any(
    r.relation_type == EvidenceRelation.CONTRADICTS and str(r.source_id) == str(closed_record.id)
    for r in contra
)
print("Contradiction keeps both records ACTIVE with a CONTRADICTS link: PASS")

# Supersede: old record marked SUPERSEDED, history preserved.
definitive = lc_store.add(ev("service retired, 443 closed", cap="teardown_audit"))
lc_store.supersede(open_record.id, definitive.id)
assert lc_store.get(open_record.id).lifecycle == EvidenceLifecycle.SUPERSEDED
assert lc_store.get(definitive.id).lifecycle == EvidenceLifecycle.ACTIVE
assert lc_store.get(open_record.id).status == EvidenceStatus.OBSERVED  # epistemic state untouched
print("Supersede marks old record without touching its status/history: PASS")

# Corroborates + typed relationship filter.
from blackforge.evidence.models import EvidenceLink as _EvidenceLink  # noqa: F401
corrob = lc_store.add(ev("independent sweep: 443 open", cap="indep_scan"))
lc_store.add_relationship(corrob.id, EvidenceRelation.CORROBORATES, definitive.id)
links = lc_store.related_evidence(definitive.id, EvidenceRelation.CORROBORATES)
assert [x.evidence.id for x in links] == [corrob.id]
print("Typed relationship lookup (CORROBORATES) returns the linked record: PASS")

# Deterministic dedup.
d1 = lc_store.add(ev("payload identical A"))
d2 = lc_store.add(ev("payload identical A"))
assert d1.id == d2.id
d3 = lc_store.add(ev("payload different B"))
assert d1.id != d3.id
print(f"Dedup: identical -> same id ({d1.id}); distinct -> separate: PASS")

print("Evidence lifecycle + relationships: PASS")

---
## 10. Evidence ↔ Memory Integration

The bridge materializes a persistent memory record that *references* evidence (never copies it). Reverse lookups answer which memory supports an evidence id and which evidence backs a memory record.

In [ ]:
from blackforge.evidence.bridge import EvidenceMemoryBridge
from blackforge.memory.manager import MemoryManager

bridge = EvidenceMemoryBridge(rules_store, MemoryManager())

# Materialize memory from evidence (status/confidence/mission/session preserved).
mem = bridge.materialize_memory(validation, memory_type="knowledge")
assert mem is not None
assert mem.evidence_ids == [validation.id], "memory must reference the evidence id"
assert mem.status == EvidenceStatus.VALIDATED, "epistemic status preserved into memory"
assert mem.confidence == validation.confidence.to_score(), "confidence score preserved"
assert mem.mission_id == MID and mem.session_id == SID, "mission/session context preserved"
assert mem.provenance.evidence_ids == [str(validation.id)]
assert mem.provenance.capability_id == "authorized_analysis"
print(f"Materialized memory {mem.id} -> references evidence {validation.id}: PASS")

# Reverse: which memory supports a given evidence?
memory_for = bridge.memory_for_evidence(validation.id)
assert any(r.id == mem.id for r in memory_for)
print("memory_for_evidence(validation.id) -> 1 record: PASS")

# Reverse: which evidence backs a given memory record?
evidence_for = bridge.evidence_for_memory(mem)
assert [e.id for e in evidence_for] == [validation.id]
assert evidence_for[0].status == EvidenceStatus.VALIDATED
print("evidence_for_memory(mem) -> returns the authoritative evidence: PASS")

# Atomic-ish create: evidence is stored first, memory second.
created_ev, created_mem = bridge.create_evidence_and_memory(
    ev("fresh finding to be remembered", cap="memorizer")
)
assert created_ev is not None and created_mem is not None
assert created_mem.evidence_ids == [created_ev.id]
print("create_evidence_and_memory writes evidence (authoritative) + linked memory: PASS")

print("Evidence <-> memory integration: PASS")

---
## 11. Restart Persistence — Evidence, Relationships & Linked Memory

Same guarantee as Phase 2, now for the evidence side: records and relationships written through one SQLite connection survive a full close/reopen on brand-new connections. This also proves the evidence↔memory link persists.

In [ ]:
from pathlib import Path
from blackforge.evidence.repository import SQLiteEvidenceRepository
from blackforge.memory.repository import SQLiteMemoryRepository

Path("data").mkdir(exist_ok=True)
EVIDENCE_DB = str(Path("data/phase3_evidence.db"))
MEMORY_DB = str(Path("data/phase3_memory.db"))
for stale in (Path(EVIDENCE_DB), Path(MEMORY_DB)):
    stale.unlink(missing_ok=True)

# ── Writer instances ────────────────────────────────────────────────────
w_store = EvidenceStore(SQLiteEvidenceRepository(EVIDENCE_DB))
w_memory = MemoryManager(SQLiteMemoryRepository(MEMORY_DB))
w_bridge = EvidenceMemoryBridge(w_store, w_memory)

w_claim = w_store.add_claim(MID, "surviving claim that will be linked", session_id=SID)
w_val = w_store.add_validation(
    MID, "example.com", "validation survives restart",
    source_capability="authorized_analysis", validates_id=w_claim.id,
)
w_mem = w_bridge.materialize_memory(w_claim)
assert w_mem is not None
w_store.close()
w_memory.close()
print("Writer closed. Records persisted.")

# ── Reader instances (brand new connections) ────────────────────────────
r_store = EvidenceStore(SQLiteEvidenceRepository(EVIDENCE_DB))
r_memory = MemoryManager(SQLiteMemoryRepository(MEMORY_DB))
r_bridge = EvidenceMemoryBridge(r_store, r_memory)

claim_loaded = r_store.get(w_claim.id)
val_loaded = r_store.get(w_val.id)
assert claim_loaded is not None and val_loaded is not None
assert claim_loaded.status == EvidenceStatus.HYPOTHESIZED
assert val_loaded.status == EvidenceStatus.VALIDATED
assert claim_loaded.lifecycle == EvidenceLifecycle.ACTIVE

validates = r_store.get_relationships(w_claim.id)
assert any(
    r.relation_type == EvidenceRelation.VALIDATES and str(r.source_id) == str(w_val.id)
    for r in validates
)
print("Evidence + VALIDATES relationship survived restart: PASS")

records = r_bridge.memory_for_evidence(w_claim.id)
assert len(records) == 1
mem_loaded = records[0]
assert mem_loaded.evidence_ids == [w_claim.id]
assert mem_loaded.status == EvidenceStatus.HYPOTHESIZED
assert mem_loaded.confidence == claim_loaded.confidence.to_score()
assert mem_loaded.provenance.capability_id == "llm_inference"

back = r_bridge.evidence_for_memory(mem_loaded)
assert [e.id for e in back] == [w_claim.id]
print("Linked memory record survived restart and resolves back to evidence: PASS")

print("Restart persistence (evidence + relationships + memory): PASS")

---
## 12. Transaction / Compensation Boundary

Evidence and memory are separate SQLite files — cross-store atomicity is NOT claimed, and the boundary is explicitly documented. Inside the bridge, however, a failed memory write leaves NO dangling memory record: the evidence stays (authoritative) and the memory side is cleaned up, so no partially-linked state remains in the process.

In [ ]:
from blackforge.memory.base import MemoryRecord

class _FailingMemory(MemoryManager):
    # Simulates a memory write that fails midway.
    def store(self, record):
        raise RuntimeError("simulated memory-store failure")

failing_bridge = EvidenceMemoryBridge(EvidenceStore(), _FailingMemory())

try:
    failing_bridge.create_evidence_and_memory(ev("should leave no dangling memory"))
    raise SystemExit("FAIL: expected simulated memory failure")
except RuntimeError as exc:
    assert "simulated memory-store failure" in str(exc)

evidence_store_for_txn = failing_bridge.evidence_store
assert evidence_store_for_txn.count() == 1, "evidence is authoritative and must remain stored"
print("Failing memory write raised; no dangling memory record (evidence remains): PASS")

print("Transaction / compensation boundary: PASS")

---
## 13. PASS/FAIL Summary

In [ ]:
RESULTS = {}

RESULTS["repository"] = (REPO_DIR / "blackforge" / "__init__.py").exists()
RESULTS["phase3_modules"] = ((REPO_DIR / "blackforge" / "evidence" / "repository.py").exists()
                             and (REPO_DIR / "blackforge" / "evidence" / "bridge.py").exists())
RESULTS["imports"] = len(_import_failures) == 0
RESULTS["tests"] = True  # Would have raised if failed
RESULTS["bootstrap"] = (app.healthy()
                        and verification["evidence_store_ready"]
                        and verification["evidence_memory_link_ready"])
RESULTS["no_fake_authority"] = (claim.status == EvidenceStatus.HYPOTHESIZED
                                and validation.status == EvidenceStatus.VALIDATED
                                and rejected and downgrade_rejected)
RESULTS["confidence_audit"] = (loaded_obs.confidence == Confidence.HIGH
                               and loaded_obs.status == EvidenceStatus.INFERRED
                               and len(loaded_obs.confidence_changes) == 1)
RESULTS["lifecycle"] = (lc_store.get(open_record.id).lifecycle == EvidenceLifecycle.SUPERSEDED
                        and lc_store.get(closed_record.id).lifecycle == EvidenceLifecycle.ACTIVE)
RESULTS["relationships"] = any(
    r.relation_type == EvidenceRelation.CORROBORATES for r in lc_store.get_relationships(definitive.id)
)
RESULTS["contradiction"] = any(
    r.relation_type == EvidenceRelation.CONTRADICTS for r in lc_store.get_relationships(open_record.id)
)
RESULTS["dedup"] = d1.id == d2.id and d1.id != d3.id
RESULTS["bridge"] = (mem.evidence_ids == [validation.id]
                     and mem.status == EvidenceStatus.VALIDATED
                     and [e.id for e in evidence_for] == [validation.id])
RESULTS["restart"] = (claim_loaded.status == EvidenceStatus.HYPOTHESIZED
                      and val_loaded.status == EvidenceStatus.VALIDATED
                      and mem_loaded.evidence_ids == [w_claim.id]
                      and [e.id for e in back] == [w_claim.id])
RESULTS["compensation"] = evidence_store_for_txn.count() == 1

print("=" * 60)
print("BLACKFORGE PHASE 3 VALIDATION (EVIDENCE <-> MEMORY INTEGRATION)")
print("=" * 60)

labels = {
    "repository": "Repository",
    "phase3_modules": "Phase 3 modules",
    "imports": "Imports",
    "tests": "Automated tests",
    "bootstrap": "Bootstrap + evidence/memory health",
    "no_fake_authority": "No fake authority (HYPOTHESIZED gate)",
    "confidence_audit": "Confidence audit (status-independent)",
    "lifecycle": "Lifecycle ops preserve history",
    "relationships": "Typed relationships",
    "contradiction": "Contradiction keeps both records",
    "dedup": "Deterministic dedup",
    "bridge": "Evidence <-> memory link",
    "restart": "Restart persistence (ev + rel + memory)",
    "compensation": "Transaction/compensation boundary",
}

for key in labels:
    status = "PASS" if RESULTS[key] else "FAIL"
    print(f"{labels[key]:<38} {status}")

print("=" * 60)

if all(RESULTS.values()):
    print("OVERALL RESULT: PASS")
else:
    failed = [labels[k] for k, v in RESULTS.items() if not v]
    print(f"OVERALL RESULT: FAIL — {', '.join(failed)}")

print("=" * 60)

In [ ]:
# Clean up: close every opened backend.
for manager in (lc_store,):
    try:
        manager.close()
    except Exception:
        pass
for manager in (r_store, r_memory):
    try:
        manager.close()
    except Exception:
        pass
print("Backends closed. Validation complete.")